# Cài đặt các Thư viện sử dụng

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import joblib
import seaborn as sns
import random
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, confusion_matrix

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec

from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB,MultinomialNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
import warnings
warnings.filterwarnings("ignore")

**Cài đặt hiển thị dataFrame**

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

**Đọc dữ liệu**

In [ ]:

file = os.path.join("../../Data_Train", "Vietnamese.csv")
data_df = pd.read_csv(file, encoding = 'utf-8')

file_stopword = os.path.join("../../Data_Train", "vietnamese-stopwords-dash.txt")
with open(file_stopword, 'r', encoding='utf-8') as file:
    stopwords = file.read().split('\n')

# Khám phá dữ liệu

**Xem thông tin giữ liệu**

In [ ]:
data_df.head(5)

**Xáo trộn dữ liệu**

In [ ]:
data_df=data_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data_df.head(5)

**Mô tả dữ liệu**

In [ ]:
data_df.info()

In [ ]:
data_df.describe().round(1)

**Kích thước của dataFrame và danh sách các tên cột**

In [ ]:
features = data_df.columns.to_list()[:]
print(data_df.shape)
print(features)

**ý nghĩa các cột**
- title: Tên của tin tức
- text: nội dung của tin tức
- label: nhãn phân biệt tin giả hay tin thật

**Xem số hàng và cột của dữ liệu**

In [ ]:
num_rows=data_df.shape[0]
num_cols=data_df.shape[1]
print(num_rows)
print(num_cols)

**Kiểm tra dữ liệu có bị lặp**

In [ ]:
dup=data_df.index.duplicated().sum()
dup

**Kiểm tra dữ liệu có bị thiếu**

In [ ]:
data_df['title'].isnull().sum()

In [ ]:
data_df['text'].isnull().sum()

In [ ]:
data_df['label'].isnull().sum()

**Kiểm tra loại dữ liệu**

In [ ]:
def open_object_dtype(s):
    dtypes = set()
    s=s.apply(type)
    dtypes.update(s.unique().tolist())
    return dtypes

In [ ]:
open_object_dtype(data_df['title'])

In [ ]:
open_object_dtype(data_df['text'])

In [ ]:
open_object_dtype(data_df['label'])

**Kiểm tra số từ của title và text**

In [ ]:
data_df["title_length"] = data_df["title"].apply(lambda title: len(title.split(" ")))
data_df["text_length"] = data_df["text"].apply(lambda text: len(text.split(" ")))

In [ ]:
data_df[["title_length", "text_length"]].describe()

In [ ]:
data_df["title_length"].plot(kind="hist", bins=20, edgecolor='black')
plt.title('Histogram of Title Length')
plt.xlabel('Title Length')
plt.ylabel('Frequency')
plt.show()

In [ ]:
data_df["text_length"].plot(kind="hist", bins=20, edgecolor='black')
plt.title('Histogram of Text Length')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

**Xem phân bố dữ liệu**

In [ ]:
# data quantity chart
def quantity_chart():
    counts_df1 = data_df['label'].value_counts()

    plt.figure(figsize=(6, 6))

    plt.pie(counts_df1, labels=counts_df1, autopct='%1.1f%%', startangle=90)
    plt.title('Bảng phân phối tin thật và tin giả')

    labels = ['Real' if label == 0 else 'Fake' for label in counts_df1.index]
    plt.legend(labels=labels, loc="best", fontsize=18)  # Tăng kích thước nhãn trong chú thích
    plt.tight_layout()
    plt.show()
quantity_chart()

**Chiều dài trung bình tiêu đề của 1 tin tức**

In [ ]:
len_title_sum=0

for i in data_df['title']:
    len_word=i.split()
    len_title_sum += len(len_word)

avg_title=len_title_sum/data_df['title'].count()

avg_title

**Độ dài lớn nhất của tiêu đề**

In [ ]:
max_title_len=0
for i in data_df['title']:
    len_word = i.split()
    if len(len_word) > max_title_len:
        max_title_len = len(len_word)
max_title_len

**Độ dài nhỏ nhất của tiêu đề**

In [ ]:
min_title_len=max_title_len

for i in data_df['title']:
    len_word=i.split()
    if len(len_word) < min_title_len:
        min_title_len = len(len_word)

min_title_len

**Chiều dài trung bình text của 1 tin tức**

In [ ]:
len_text_sum=0

for i in data_df['text']:
    len_word=i.split()
    len_text_sum += len(len_word)

avg_text=len_text_sum/data_df['text'].count()

avg_text

**Độ dài lớn nhất của text**

In [ ]:
max_text_len=0

for i in data_df['text']:
    len_word=i.split()
    if len(len_word)>max_text_len:
        max_text_len=len(len_word)

max_text_len

**Độ dài nhỏ nhất của text**

In [ ]:
min_text_len = max_text_len

for i in data_df['text']:
    len_word=i.split()
    if len(len_word) < min_text_len:
        min_text_len = len(len_word)

min_text_len

**Tổng số từ trong title**

In [ ]:
total_words = data_df['title'].str.split().map(len).sum()
total_words

**Tổng số từ trong text**

In [ ]:
total_words = data_df['text'].str.split().map(len).sum()
total_words

**Số từ khác biệt trong title**

In [ ]:
unique_words = len(set(' '.join(data_df['title']).split()))
unique_words

**Số từ khác biệt trong text**

In [ ]:
unique_words = len(set(' '.join(data_df['text']).split()))
unique_words

# Tiền xử lí văn bản

#### Loại bỏ các đường link và các dấu câu, lowercase

**Thư viện để xử lý**

In [ ]:
from pyvi import ViTokenizer

**Loại bỏ các đường link và các dấu câu, lowercase**

In [ ]:
def wordopt(text):
    text = text.lower()
    text = re.sub('[%s]' % re.escape("""!–"#$%&'()*+,،-./:;<=>؟?@[\]^`{|}~“”…؛"""), ' ', text)
    text = re.sub('https?://\S+|www\.\S+|https?:\/\/.*[\r\n]*', ' ', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('\\W', ' ', text)
    text = re.sub('\w*\d\w*', '', text)
    text = re.sub('\n', ' ', text)
    text = re.sub('\s+', ' ', text)
    text = text.strip()

    update_text = ""
    for word in text.split():
        if word not in stopwords:
            update_text += word+" "
        
    return update_text.strip()

In [ ]:
# Compound Vietnamese word
def tokenizerVN(text):
    return ViTokenizer.tokenize(text)

**Hợp nhất title và text**

In [ ]:
data_df['article'] = data_df["title"] +  data_df["text"] 

In [ ]:
data_df.head(1)

**Áp dụng các xử lý dữ liệu**

In [ ]:
data_df['Compound_Content'] = data_df['article'].apply(tokenizerVN)

In [ ]:
data_df['Compound_Content_SW'] = data_df['Compound_Content'].apply(preprocess_nostop)

In [ ]:
data_df.head(1)

**Tổng số từ**

In [ ]:
total_words = data_df['Compound_Content_SW'].str.split().map(len).sum()
total_words

**Số từ khác biệt còn lại**

In [ ]:
unique_words_combined = len(set(' '.join(data_df['Compound_Content_SW']).split()))
unique_words_combined

**Xác định tần suất xuất hiện các từ trong từng tài liệu**

In [ ]:
from collections import Counter

In [ ]:
word_counts_per_document = []

# Duyệt qua từng tài liệu
for doc in data_df['Compound_Content_SW']:
    words_in_doc = set(doc.split())  # Sử dụng set để loại bỏ từ trùng trong mỗi tài liệu
    word_counts_per_document.append(words_in_doc)

# Bước 4: Đếm tổng số lần xuất hiện của các từ trong tất cả các tài liệu (mỗi từ chỉ tính một lần trong một tài liệu)
total_word_counts = Counter()

# Duyệt qua danh sách các từ trong từng tài liệu và cập nhật tổng số đếm
for word_set in word_counts_per_document:
    total_word_counts.update(word_set)

In [ ]:
print(len(total_word_counts))
for word, count in total_word_counts.most_common():
    print(f"{word}: {count/len(data_df)}")


# Vector hóa

## Chuẩn bị

**Tokenizer Tách câu thành từ**

In [ ]:
def tokenize(sentence):
    return tokenizerVN(sentence).split()


**Chia tập dữ liệu thành 2 tập**

- Tập train: 75%
- Tập test: 25%


In [ ]:
X = data_df['Compound_Content_SW']
y = data_df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25,
                                                   random_state=30)

**Các đường dẫn để lưu model**

In [ ]:
model_vector_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_vectorizer_CV.joblib")
model_DTC_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_DTC_model_CV.joblib")
model_NB_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_NB_model_CV.joblib")
model_RFC_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_RFC_model_CV.joblib")
model_SVM_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_SVM_model_CV.joblib")
model_LR_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_LR_model_CV.joblib")
model_GBC_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_GBC_model_CV.joblib")


## Các mô hình vector

**Vẽ hình**

In [ ]:
color_names = [
    "Blues", "Purples", "Greens", "Oranges", "Reds",
    "coolwarm", "cubehelix", "YlGnBu", "BuPu", "GnBu"
]
model_colors = []

**Tính toán đánh giá**

In [ ]:
choose_color = "Blues"
def get_model_color():
    for pick_color in color_names:
        if pick_color not in model_colors:
            model_colors.append(pick_color)
            return pick_color
    return "Blues"

In [ ]:
def evaluate_model(y_true, y_pred, name_model, save=False):

    cm = confusion_matrix(y_true, y_pred)    
    
    selected_palette = get_model_color() 
    plt.figure(figsize=(6, 4)) 
    sns.heatmap(cm, cmap=selected_palette, annot=True, fmt="d", cbar=True) 
    plt.title(f"Confusion Matrix - {name_model}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


    # Tỷ lệ mẫu Positive dự đoán đúng trên tất cả mẫu dự đoán Positive.
    PPV = precision_score(y_true, y_pred)

    # Tỷ lệ mẫu Positive được phát hiện đúng.
    TPR = recall_score(y_true, y_pred)
    
    # Tỷ lệ dự đoán đúng (cả Positive và Negative).
    ACC = accuracy_score(y_true, y_pred)
    # Trung bình điều hòa giữa Precision và Recall.
    F1 = f1_score(y_true, y_pred)
    
    # Tỷ lệ mẫu Negative bị nhầm thành Positive.
    FPR = cm[0][1] / (cm[0][0] + cm[0][1])
    # Tỷ lệ mẫu Positive bị nhầm thành Negative.
    FNR = cm[1][0] / (cm[1][0] + cm[1][1])

    metrics = {
        "Precision (PPV)": PPV,
        "Recall (TPR)": TPR,
        "Accuracy (ACC)": ACC,
        "F1-Score": F1,
        "False Positive Rate (FPR)": FPR,
        "False Negative Rate (FNR)": FNR,
    }
    if save:
        metrics_df = pd.DataFrame(metrics.items(), columns=["evaluation metrics", "Value"])
        path_evaluate = os.path.join("../../Evaluate/English", f"English_Evaluate_{name_model}.csv")
        
        metrics_df.to_csv(path_evaluate, index=False)

### CountVectorizer

In [ ]:
vectorizerCV = CountVectorizer(
                            stop_words=stopwords,
                            tokenizer=tokenize,
                            min_df=0.001,          
                            max_df=0.5,            
                            max_features=12000,              
                            ngram_range=(1, 3)        
                            )

In [ ]:
Xv_trainCV = vectorizerCV.fit_transform(X_train)
Xv_testCV = vectorizerCV.transform(X_test)

In [ ]:
joblib.dump(vectorizerCV, model_vector_CV)

**DecisionTreeClassifier**

In [ ]:
dtc_cv = DecisionTreeClassifier()
dtc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_dt_cv = dtc_cv.predict(Xv_testCV)
accuracy = accuracy_score(y_test, pred_dt_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_dt_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_dt_cv,name_model="dt_cv")

In [ ]:
joblib.dump(dtc_cv, model_DTC_CV)

### Navie bayes

In [ ]:
nb_cv = MultinomialNB()
nb_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_nb_cv = nb_cv.predict(Xv_testCV)

accuracy = accuracy_score(y_test, pred_nb_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_nb_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_nb_cv,name_model="nb_cv")

In [ ]:
joblib.dump(nb_cv, model_NB_CV)

### RandomForest

In [ ]:
rfc_cv = RandomForestClassifier()
rfc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_rfc_cv = rfc_cv.predict(Xv_testCV)

accuracy = accuracy_score(y_test, pred_rfc_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_rfc_cv,name_model="rfc_cv")

In [ ]:
joblib.dump(rfc_cv, model_RFC_CV)

### SVM

In [ ]:
svm_cv = LinearSVC(random_state=42)
svm_cv.fit(Xv_trainCV, y_train)

In [ ]:
pred_svm_cv = svm_cv.predict(Xv_testCV)

accuracy = accuracy_score(y_test, pred_svm_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_svm_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_rfc_cv,name_model="svm_cv")

In [ ]:
joblib.dump(svm_cv, model_SVM_CV)

### LogisticRegression

In [ ]:
lr_cv = LogisticRegression()
lr_cv.fit(Xv_trainCV, y_train)
pred_lr_cv = lr_cv.predict(Xv_testCV)

In [ ]:
accuracy = accuracy_score(y_test, pred_lr_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_lr_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_lr_cv,name_model="lr_cv")

In [ ]:
joblib.dump(lr_cv, model_LR_CV)

### GradientBoostingClassifier

In [ ]:
gbc_cv = GradientBoostingClassifier(random_state=0)
gbc_cv.fit(Xv_trainCV, y_train)
pred_gbc_cv = gbc_cv.predict(Xv_testCV)

In [ ]:
accuracy = accuracy_score(y_test, pred_gbc_cv) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_gbc_cv))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_lr_cv,name_model="gbc_cv")

In [ ]:
joblib.dump(gbc_cv, model_GBC_CV)

## TfidfVectorizer

In [ ]:
model_vector_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_vectorizer_TF.joblib")
model_DTC_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_DTC_model_TF.joblib")
model_NB_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_NB_model_TF.joblib")
model_RFC_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_RFC_model_TF.joblib")
model_SVM_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_SVM_model_TF.joblib")
model_LR_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_LR_model_TF.joblib")
model_GBC_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_GBC_model_TF.joblib")

In [ ]:
vectorizerTF = TfidfVectorizer(
                            stop_words=stopwords,
                            tokenizer=tokenize,
                            min_df=0.001,          
                            max_df=0.5,            
                            max_features=12000, 
                            use_idf=True, 
                            ngram_range=(1,3)
                            )

In [ ]:
Xv_trainTF = vectorizerTF.fit_transform(X_train)
Xv_testTF = vectorizerTF.transform(X_test)

In [ ]:
joblib.dump(vectorizerTF, model_vector_TF)

### DecisionTreeClassifier

In [ ]:
dtc_tf = DecisionTreeClassifier()
dtc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_dtc_tf = dtc_tf.predict(Xv_testTF)

In [ ]:
accuracy = accuracy_score(y_test, pred_dtc_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_dtc_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_dtc_tf,name_model="dt_tf")

In [ ]:
joblib.dump(dtc_tf, model_DTC_TF)

### Navie bayes

In [ ]:
nb_tf = MultinomialNB()
nb_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_nb_tf = nb_tf.predict(Xv_testTF)

In [ ]:
accuracy = accuracy_score(y_test, pred_nb_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_nb_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_nb_tf,name_model="nb_tf")

In [ ]:
#save model Naive Bayes
joblib.dump(nb_tf, model_NB_TF)

### RandomForest

In [ ]:
rfc_tf = RandomForestClassifier()
rfc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_rfc_tf = rfc_tf.predict(Xv_testTF)

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_rfc_tf,name_model="rfc_tf")

In [ ]:
joblib.dump(rfc_tf, model_RFC_TF)

### SVM

In [ ]:
svm_tf = LinearSVC(random_state=42)
svm_tf.fit(Xv_trainTF, y_train)

In [ ]:
pred_svm_tf = svm_tf.predict(Xv_testTF)

In [ ]:
accuracy = accuracy_score(y_test, pred_svm_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_svm_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_svm_tf,name_model="svm_tf")

In [ ]:
joblib.dump(svm_tf, model_SVM_TF)

### LogisticRegression

In [ ]:
lr_tf = LogisticRegression()
lr_tf.fit(Xv_trainTF, y_train)
pred_lr_tf = lr_tf.predict(Xv_testTF)

In [ ]:
accuracy = accuracy_score(y_test, pred_lr_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_lr_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_lr_tf,name_model="lr_tf")

### GradientBoostingClassifier

In [ ]:
gbc_tf = GradientBoostingClassifier(random_state=0)
gbc_tf.fit(Xv_trainTF, y_train)
pred_gbc_tf = gbc_tf.predict(Xv_testTF)


In [ ]:
accuracy = accuracy_score(y_test, pred_gbc_tf) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_gbc_tf))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_gbc_tf,name_model="gbc_tf")

## Word2Vec

In [ ]:
model_vector_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_vectorizer_W2V.joblib")
model_DTC_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_DTC_model_W2V.joblib")
model_NB_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_NB_model_W2V.joblib")
model_RFC_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_RFC_model_W2V.joblib")
model_SVM_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_SVM_model_W2V.joblib")
model_LR_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_LR_model_W2V.joblib")
model_GBC_W2V = os.path.join("../../Model/Vietnamese/W2V", "Vietnamese_GBC_model_W2V.joblib")

In [ ]:
def makeWords(sentences):
    wordList = []
    for sentence in sentences:
        words = sentence.split(' ')
        wordList.append(words)
    return wordList

words_train_W2V = makeWords(X_train)
words_test_W2V = makeWords(X_test)

In [ ]:
# Train the Word2Vec model on the tokenized training sentences
vectorizerW2V = Word2Vec(
    sentences=words_train_W2V,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,
    epochs=20
)

In [ ]:
def sentence_vector(sentence, model):
    vectors = [model.wv[word] for word in sentence if word in model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

In [ ]:
Xv_trainW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_train_W2V])
Xv_testW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_test_W2V])

In [ ]:
joblib.dump(vectorizerW2V,model_vector_W2V)

### Decision Tree

In [ ]:
dtc_w2v = DecisionTreeClassifier()
dtc_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_dtc_w2v = dtc_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_dtc_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_dtc_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_dtc_w2v,name_model="dtc_w2v")

In [ ]:
joblib.dump(dtc_w2v, model_DTC_W2V)

### Navie bayes

In [ ]:
nb_w2v = GaussianNB()
nb_w2v.fit(Xv_trainW2V,y_train)

In [ ]:
pred_np_w2v = nb_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_np_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_np_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_np_w2v,name_model="nb_w2v")

In [ ]:
joblib.dump(nb_w2v, model_NB_W2V)

### RandomForest

In [ ]:
rfc_w2v = RandomForestClassifier()
rfc_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_rfc_w2v = rfc_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_rfc_w2v,name_model="rfc_w2v")

In [ ]:
joblib.dump(rfc_w2v, model_RFC_W2V)

### SVM

In [ ]:
svm_w2v = LinearSVC(random_state=42)
svm_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_svm_w2v = svm_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_svm_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_svm_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_svm_w2v,name_model="svm_w2v")

In [ ]:
joblib.dump(svm_w2v, model_SVM_W2V)

### LogisticRegression

In [ ]:
lr_w2v = LogisticRegression()
lr_w2v.fit(Xv_trainW2V, y_train)
pred_lr_w2v = lr_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_lr_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_lr_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_lr_w2v,name_model="lr_w2v")

In [ ]:
joblib.dump(lr_w2v, model_LR_W2V)

### GradientBoostingClassifier

In [ ]:
gbc_w2v = GradientBoostingClassifier(random_state=0)
gbc_w2v.fit(Xv_trainW2V, y_train)
pred_gbc_w2v = gbc_w2v.predict(Xv_testW2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_gbc_w2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_gbc_w2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_gbc_w2v,name_model="gbc_w2v")

In [ ]:
joblib.dump(gbc_w2v, model_GBC_W2V)

## Doc2Vec

In [ ]:
model_vector_D2V = os.path.join("../../Model/Vietnamese/D2V", "Vietnamese_vectorizer_D2V.joblib")
model_DTC_D2V = os.path.join("../../Model/Vietnamese/D2V", "Vietnamese_DTC_model_D2V.joblib")
model_NB_D2V = os.path.join("../../Model/Vietnamese/D2V", "Vietnamese_NB_model_D2V.joblib")
model_RFC_D2V = os.path.join("../../Model/Vietnamese/D2V", "Vietnamese_RFC_model_D2V.joblib")
model_SVM_D2V = os.path.join("../../Model/Vietnamese/D2V", "Vietnamese_SVM_model_D2V.joblib")
model_LR_D2V = os.path.join("../../Model/VietNamese/D2V", "VietNamese_LR_model_D2V.joblib")
model_GBC_D2V = os.path.join("../../Model/VietNamese/D2V", "VietNamese_GBC_model_D2V.joblib")

In [ ]:
def preprocess(doc):
    tokens = tokenize(doc)
    return [word for word in tokens if word not in stopwords]

In [ ]:
documents = [TaggedDocument(preprocess(doc), [i]) for i, doc in enumerate(X_train)]

In [ ]:
vectorizerD2V = Doc2Vec(
    documents=documents,  # Truyền documents trực tiếp
    vector_size=100,      # Kích thước vector
    window=5,             # Kích thước cửa sổ ngữ cảnh
    min_count=1,          # Bỏ qua từ xuất hiện ít hơn min_count
    workers=4,            # Số luồng để huấn luyện
    epochs=20             # Số lần lặp để huấn luyện
)

In [ ]:
Xv_trainD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_train]
Xv_testD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_test]

In [ ]:
joblib.dump(vectorizerD2V, model_vector_D2V)

### DecisionTreeClassifier

In [ ]:
dtc_d2v = DecisionTreeClassifier()
dtc_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_dtc_d2v = dtc_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_dtc_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_dtc_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_dtc_d2v,name_model="dtc_d2v",save=True)

In [ ]:
joblib.dump(dtc_d2v, model_DTC_D2V)

### Navie bayes

In [ ]:
nb_d2v = GaussianNB()
nb_d2v.fit(Xv_trainD2V,y_train)

In [ ]:
pred_nb_d2v = nb_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_nb_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_nb_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_nb_d2v,name_model="nb_d2v",save=True)

In [ ]:
joblib.dump(nb_d2v, model_NB_D2V)

### RandomForest

In [ ]:
rfc_d2v = RandomForestClassifier()
rfc_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_rfc_d2v = rfc_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_rfc_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_rfc_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_rfc_d2v,name_model="rfc_d2v",save=True)

In [ ]:
joblib.dump(rfc_d2v, model_RFC_D2V)

### SVM

In [ ]:
svm_d2v = LinearSVC(random_state=42)
svm_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_svm_d2v = svm_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_svm_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_svm_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_svm_d2v,name_model="smv_d2v",save=True)

In [ ]:
joblib.dump(svm_d2v, model_SVM_D2V)

### LogisticRegression

In [ ]:
lr_d2v = LogisticRegression()
lr_d2v.fit(Xv_trainD2V, y_train)
pred_lr_d2v = lr_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_lr_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_lr_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_lr_d2v,name_model="lr_d2v")

In [ ]:
joblib.dump(lr_d2v, model_LR_D2V)

### GradientBoostingClassifier

In [ ]:
gbc_d2v = GradientBoostingClassifier(random_state=0)
gbc_d2v.fit(Xv_trainD2V, y_train)
pred_gbc_d2v = gbc_d2v.predict(Xv_testD2V)

In [ ]:
accuracy = accuracy_score(y_test, pred_gbc_d2v) 
print(accuracy)
print('-------------------\n')

print(classification_report(y_true=y_test, y_pred=pred_gbc_d2v))
print('-------------------\n')

evaluate_model(y_true=y_test, y_pred=pred_gbc_d2v,name_model="gbc_d2v")

In [ ]:
joblib.dump(gbc_d2v, model_GBC_D2V)